# Yoga Pose Detection — Pose Processing Project
Classifies a photo into one of the yoga poses in the dataset (Downdog, Goddess, Plank, Tree, Warrior2) by training a
new CNN-feature-based classifier on real labeled photos.

**Before running:** unzip the Teachable Machine model files once so `keras_model.h5`, `metadata.json` and `weights.bin`
exist on disk, then set the paths in the **Configuration** cells below.

## Configuration

In [ ]:
# Paths (change these to match your folders)
MASK_MODEL_PATH = "models/keras_model.h5"
POSE_METADATA_PATH = "models/metadata.json"
POSE_WEIGHTS_PATH = "models/weights.bin"
POSE_DATASET_DIR = "POSE_DATASET"
SAVED_MODEL_PATH = "pose_classifier.keras"

# Dataset sub-folders (each one contains one folder per pose)
TRAIN_DIR = f"{POSE_DATASET_DIR}/TRAIN"
TEST_DIR = f"{POSE_DATASET_DIR}/TEST"

In [ ]:
# Data settings
TRAIN_PER_CLASS = 100
TEST_PER_CLASS = 40
VAL_SIZE = 0.2
RANDOM_SEED = 42

# Pose classifier settings
CLASSIFIER_UNITS = 64
CLASSIFIER_DROPOUT = 0.3
EPOCHS = 15
BATCH_SIZE = 16
FEATURE_BATCH_SIZE = 32

# Layout of the pretrained Teachable Machine pose model
POSE_HIDDEN_UNITS = 100
POSE_DROPOUT = 0.5

# Number of feature maps to display in section 7
NUM_FILTERS_TO_SHOW = 5

## 1. Import Libraries

In [ ]:
import os
import json
import glob
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

ImageFile.LOAD_TRUNCATED_IMAGES = True   # do not fail on slightly damaged photos

In [ ]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # hide TensorFlow info messages

import tensorflow as tf
import tf_keras as keras
from tf_keras.layers import DepthwiseConv2D

In [ ]:
warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

## 2. Load the Pretrained Pose Model (for reference) & the CNN Feature Extractor

In [ ]:
# Pretrained Teachable Machine pose model: classifier head only, its input is PoseNet keypoints
with open(POSE_METADATA_PATH) as file:
    pose_meta = json.load(file)

pose_labels = pose_meta["labels"]
weights = np.fromfile(POSE_WEIGHTS_PATH, dtype=np.float32)

In [ ]:
# Work out the layer sizes from the weights file instead of typing them
num_labels = len(pose_labels)
input_size = (weights.size - POSE_HIDDEN_UNITS) // POSE_HIDDEN_UNITS - num_labels

# The weights file holds: layer 1 weights, layer 1 bias, layer 2 weights (in this order)
end_w1 = input_size * POSE_HIDDEN_UNITS
end_b1 = end_w1 + POSE_HIDDEN_UNITS

w1 = weights[:end_w1].reshape(input_size, POSE_HIDDEN_UNITS)
b1 = weights[end_w1:end_b1]
w2 = weights[end_b1:].reshape(POSE_HIDDEN_UNITS, num_labels)

In [ ]:
tm_pose_model = keras.Sequential([
    keras.layers.Dense(POSE_HIDDEN_UNITS, activation="relu", input_shape=(input_size,)),
    keras.layers.Dropout(POSE_DROPOUT),
    keras.layers.Dense(num_labels, activation="softmax", use_bias=False),
])
tm_pose_model.layers[0].set_weights([w1, b1])
tm_pose_model.layers[2].set_weights([w2])

print("TM pose model -> input:", tm_pose_model.input_shape, " output:", tm_pose_model.output_shape)

In [ ]:
# The old Teachable Machine export passes an argument ("groups") that newer Keras rejects
class DepthwiseConv2DFix(DepthwiseConv2D):
    def __init__(self, **kwargs):
        kwargs.pop("groups", None)
        super().__init__(**kwargs)

In [ ]:
# CNN feature extractor: the MobileNet backbone of the mask model, it works directly on photos
mask_model = keras.models.load_model(
    MASK_MODEL_PATH,
    compile=False,
    custom_objects={"DepthwiseConv2D": DepthwiseConv2DFix},
)

feature_extractor = mask_model.layers[0]
feature_extractor.trainable = False

IMAGE_SIZE = feature_extractor.input_shape[1:3]   # (height, width) read from the model itself

print("CNN feature extractor -> input:", feature_extractor.input_shape, " output:", feature_extractor.output_shape)

## 3. Prepare & Preprocess the Data (Real Labeled Photos)

In [ ]:
# One class per folder inside TRAIN (sorted, so the class order is always the same)
classes = []
for name in sorted(os.listdir(TRAIN_DIR)):
    if os.path.isdir(f"{TRAIN_DIR}/{name}"):
        classes.append(name)
        assert os.path.isdir(f"{TEST_DIR}/{name}"), f"Missing TEST folder for class: {name}"

print("Classes:", classes)

In [ ]:
def get_image_paths(folder, limit=None):
    paths = sorted(glob.glob(f"{folder}/*"))
    return paths[:limit]


def load_split(split_dir, per_class):
    paths = []
    labels = []
    for class_index, class_name in enumerate(classes):
        class_paths = get_image_paths(f"{split_dir}/{class_name}", per_class)
        paths.extend(class_paths)
        labels.extend([class_index] * len(class_paths))
    return paths, np.array(labels)

In [ ]:
def preprocess_image(path):
    image = Image.open(path).convert("RGB")
    image = image.resize(IMAGE_SIZE)
    array = np.asarray(image, dtype=np.float32)
    return array / 127.5 - 1   # scale pixels from 0-255 to -1..1, as the model expects


def load_images(paths):
    images = []
    for path in paths:
        images.append(preprocess_image(path))
    return np.array(images)

In [ ]:
train_paths, y_train = load_split(TRAIN_DIR, TRAIN_PER_CLASS)
test_paths, y_test = load_split(TEST_DIR, TEST_PER_CLASS)

for class_index, class_name in enumerate(classes):
    print(class_name, ": train", np.sum(y_train == class_index), " test", np.sum(y_test == class_index))

In [ ]:
fig, axes = plt.subplots(1, len(classes), figsize=(3 * len(classes), 3.2))

for class_index, class_name in enumerate(classes):
    first_position = np.where(y_train == class_index)[0][0]
    axes[class_index].imshow(Image.open(train_paths[first_position]))
    axes[class_index].set_title(class_name)
    axes[class_index].axis("off")

plt.show()

In [ ]:
X_train_img = load_images(train_paths)
X_test_img = load_images(test_paths)

print("Train:", X_train_img.shape, " Test:", X_test_img.shape)

## 4. Integrate — Extract CNN Features

In [ ]:
feat_train = feature_extractor.predict(X_train_img, verbose=0, batch_size=FEATURE_BATCH_SIZE)
feat_test = feature_extractor.predict(X_test_img, verbose=0, batch_size=FEATURE_BATCH_SIZE)

print("Feature shape:", feat_train.shape)

## 5. Train a New Pose Classifier

In [ ]:
# Keep some training photos aside for validation, so the test photos stay unseen until section 6
X_fit, X_val, y_fit, y_val = train_test_split(
    feat_train, y_train, test_size=VAL_SIZE, random_state=RANDOM_SEED, stratify=y_train
)
print("Fit:", X_fit.shape, " Validation:", X_val.shape)

In [ ]:
pose_classifier = keras.Sequential([
    keras.layers.Input(shape=(feat_train.shape[1],)),
    keras.layers.Dense(CLASSIFIER_UNITS, activation="relu"),
    keras.layers.Dropout(CLASSIFIER_DROPOUT),
    keras.layers.Dense(len(classes), activation="softmax"),
])

pose_classifier.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [ ]:
history = pose_classifier.fit(
    X_fit, y_fit,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=0,
)

In [ ]:
print("Train accuracy:", round(history.history["accuracy"][-1], 3))
print("Val accuracy:  ", round(history.history["val_accuracy"][-1], 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

for ax, metric in zip(axes, ["accuracy", "loss"]):
    ax.plot(history.history[metric], label="Train")
    ax.plot(history.history["val_" + metric], label="Validation")
    ax.set_title(metric.capitalize())
    ax.set_xlabel("Epoch")
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Evaluate on Real Held-Out Test Photos

In [ ]:
test_preds = pose_classifier.predict(feat_test, verbose=0).argmax(axis=1)

In [ ]:
print("Test accuracy:", round(accuracy_score(y_test, test_preds), 3))
print()
print(classification_report(y_test, test_preds, target_names=classes))

In [ ]:
cm = confusion_matrix(y_test, test_preds)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 7. CNN Concept: What the Convolution Layers See
A look inside the CNN — the feature maps an early convolution layer produces for one real pose photo.

In [ ]:
cnn_backbone = feature_extractor.layers[0]   # the MobileNet CNN inside the feature extractor

for layer in cnn_backbone.layers:
    if isinstance(layer, keras.layers.Conv2D):
        first_conv_layer = layer
        break

print("First Conv2D layer:", first_conv_layer.name)

In [ ]:
feature_map_model = keras.Model(cnn_backbone.input, first_conv_layer.output)

sample_img = X_test_img[0:1]
feature_maps = feature_map_model.predict(sample_img, verbose=0)[0]

In [ ]:
fig, axes = plt.subplots(1, NUM_FILTERS_TO_SHOW + 1, figsize=(2 * (NUM_FILTERS_TO_SHOW + 1), 2.4))

axes[0].imshow((sample_img[0] + 1) / 2)   # undo the -1..1 scaling to show the photo
axes[0].set_title("Input")

for i in range(NUM_FILTERS_TO_SHOW):
    axes[i + 1].imshow(feature_maps[:, :, i], cmap="viridis")
    axes[i + 1].set_title(f"Filter {i + 1}")

for ax in axes:
    ax.axis("off")

plt.suptitle(f'Feature maps from "{first_conv_layer.name}"')
plt.tight_layout()
plt.show()

## 8. Save & Reload the Trained Model

In [ ]:
pose_classifier.save(SAVED_MODEL_PATH)
print("Model saved to:", SAVED_MODEL_PATH)

In [ ]:
loaded_pose_classifier = keras.models.load_model(SAVED_MODEL_PATH)
print("Reloaded model input shape:", loaded_pose_classifier.input_shape)

## 9. Test on New Photos

In [ ]:
def predict_pose(image_path, extractor, classifier):
    image = preprocess_image(image_path)
    features = extractor.predict(np.expand_dims(image, axis=0), verbose=0)
    probabilities = classifier.predict(features, verbose=0)[0]
    best = int(np.argmax(probabilities))
    return classes[best], round(float(probabilities[best]) * 100, 2)

In [ ]:
# For each pose, take the first test photo that was NOT used in the evaluation
fig, axes = plt.subplots(1, len(classes), figsize=(3.2 * len(classes), 4.2))

for class_index, class_name in enumerate(classes):
    paths = get_image_paths(f"{TEST_DIR}/{class_name}")
    new_image = paths[min(TEST_PER_CLASS, len(paths) - 1)]
    label, confidence = predict_pose(new_image, feature_extractor, loaded_pose_classifier)

    axes[class_index].imshow(Image.open(new_image))
    axes[class_index].set_title(f"Actual: {class_name}\nPredicted: {label}\n({confidence}%)")
    axes[class_index].axis("off")

plt.tight_layout()
plt.show()